# Lab 10 — Pipeline Definitivo (RAG + QLoRA + Otimização GPU)

> Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por José Arthur 

In [ ]:
# Célula 1 — Instalação
!pip install -q transformers bitsandbytes accelerate flash-attn --no-build-isolation

In [ ]:
# Célula 2 — Imports
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Célula 3 — Carregamento quantizado
torch.cuda.reset_peak_memory_stats()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Métrica obrigatória — anote no README
vram_carregamento = torch.cuda.max_memory_allocated() / 1024**2
print(f"VRAM após carregamento: {vram_carregamento:.2f} MB")

In [ ]:
# Célula 4 — Geração do contexto fictício (~15.000 tokens)
paragrafo_medico = (
    "The patient presented with acute respiratory distress syndrome. "
    "Chest X-ray revealed bilateral infiltrates consistent with pneumonia. "
    "SpO2 was 88% on room air, requiring supplemental oxygen at 4L/min. "
    "Laboratory results showed elevated CRP, leukocytosis, and elevated D-dimer. "
    "The clinical team initiated broad-spectrum antibiotics and anticoagulation therapy. "
) * 200  # Repetir para atingir volume de tokens

inputs = tokenizer(paragrafo_medico, return_tensors="pt", truncation=True, max_length=12000).to(DEVICE)
print(f"Tokens no contexto: {inputs['input_ids'].shape[1]}")

In [ ]:
# Célula 5 — Geração sem cache (baseline lento)
model.config.use_cache = False
torch.cuda.reset_peak_memory_stats()

inicio = time.time()

with torch.no_grad():
    output_sem_cache = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=False
    )

tempo_sem_cache = time.time() - inicio
vram_sem_cache = torch.cuda.max_memory_allocated() / 1024**2

print(f"[SEM CACHE] Tempo: {tempo_sem_cache:.2f}s | VRAM pico: {vram_sem_cache:.2f} MB")

In [ ]:
# Célula 6 — Recarregar modelo com FlashAttention-2 (fallback: SDPA do PyTorch)
# FlashAttention-2 exige GPU Ampere+ (A100, A10G). T4 do Colab free não suporta.
# SDPA (Scaled Dot Product Attention) funciona em qualquer GPU e dá resultado equivalente.
del model
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

try:
    model_otimizado = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="flash_attention_2"
    )
    impl_usada = "flash_attention_2"
except Exception as e:
    print(f"FlashAttention-2 indisponível (GPU não é Ampere+): {e}\nUsando SDPA (PyTorch).")
    model_otimizado = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="sdpa"
    )
    impl_usada = "sdpa"

model_otimizado.config.use_cache = True
print(f"Implementação de atenção: {impl_usada}")

In [ ]:
# Célula 7 — Geração otimizada
torch.cuda.reset_peak_memory_stats()

inicio = time.time()

with torch.no_grad():
    output_otimizado = model_otimizado.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=True
    )

tempo_com_cache = time.time() - inicio
vram_com_cache = torch.cuda.max_memory_allocated() / 1024**2

print(f"[COM CACHE + {impl_usada.upper()}] Tempo: {tempo_com_cache:.2f}s | VRAM pico: {vram_com_cache:.2f} MB")
print(f"\nSpeedup: {tempo_sem_cache / tempo_com_cache:.2f}x mais rápido")
print(f"Redução VRAM: {vram_sem_cache - vram_com_cache:.2f} MB")

In [ ]:
# Célula 8 — Resumo das métricas
print("=" * 55)
print(f"{'Métrica':<30} {'Sem Cache':>10} {'Otimizado':>12}")
print("=" * 55)
print(f"{'VRAM carregamento (MB)':<30} {vram_carregamento:>10.1f} {vram_carregamento:>12.1f}")
print(f"{'VRAM pico geração (MB)':<30} {vram_sem_cache:>10.1f} {vram_com_cache:>12.1f}")
print(f"{'Tempo de geração (s)':<30} {tempo_sem_cache:>10.2f} {tempo_com_cache:>12.2f}")
print(f"{'Implementação atenção':<30} {'standard':>10} {impl_usada:>12}")
print("=" * 55)